In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Configuration du device (GPU si disponible)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Utilisation du device: {device}")

# ==================== ÉTAPE 1: CHARGEMENT ET PRÉTRAITEMENT ====================

def load_and_preprocess_data(ticker='AAPL', start_date='2020-01-01', end_date='2023-12-31'):
    """
    Télécharge et prétraite les données boursières
    """
    print(f"Téléchargement des données pour {ticker}...")
    
    # Téléchargement des données
    df = yf.download(ticker, start=start_date, end=end_date)
    
    # Création de la colonne cible (prix de clôture du lendemain)
    df['target'] = df['Close'].shift(-1)
    
    # Suppression des lignes avec des valeurs NaN
    df = df.dropna()
    
    # Sélection des features
    features = ['Open', 'High', 'Low', 'Close', 'Volume']
    X = df[features].values
    y = df['target'].values.reshape(-1, 1)
    
    print(f"Données chargées: {len(df)} échantillons")
    print(f"Période: de {df.index[0]} à {df.index[-1]}")
    
    return X, y, df.index

# ==================== ÉTAPE 2: NORMALISATION ====================

def normalize_data(X, y):
    """
    Normalise les données avec MinMaxScaler
    """
    scaler_X = MinMaxScaler()
    scaler_y = MinMaxScaler()
    
    X_scaled = scaler_X.fit_transform(X)
    y_scaled = scaler_y.fit_transform(y)
    
    print(f"Données normalisées - X shape: {X_scaled.shape}, y shape: {y_scaled.shape}")
    
    return X_scaled, y_scaled, scaler_X, scaler_y

# ==================== ÉTAPE 3: SÉQUENCES TEMPORELLES ====================

def create_sequences(X, y, seq_length=60):
    """
    Crée des séquences temporelles pour l'entraînement LSTM
    """
    X_seq, y_seq = [], []
    
    for i in range(seq_length, len(X)):
        X_seq.append(X[i-seq_length:i])
        y_seq.append(y[i])
    
    return np.array(X_seq), np.array(y_seq)

# ==================== ÉTAPE 4: DATASET PYTORCH PERSONNALISÉ ====================

class StockDataset(Dataset):
    """
    Dataset personnalisé pour les données boursières
    """
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# ==================== ÉTAPE 5: MODÈLE LSTM ====================

class LSTMPredictor(nn.Module):
    """
    Modèle LSTM avec couches additionnelles
    """
    def __init__(self, input_size, hidden_size=128, num_layers=2, dropout=0.2):
        super(LSTMPredictor, self).__init__()
        
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )
        
        self.dropout = nn.Dropout(dropout)
        self.fc1 = nn.Linear(hidden_size, 64)
        self.fc2 = nn.Linear(64, 1)
        self.relu = nn.ReLU()
        
    def forward(self, x):
        # LSTM layers
        lstm_out, _ = self.lstm(x)
        
        # Prendre la dernière sortie de la séquence
        last_output = lstm_out[:, -1, :]
        
        # Couches denses
        x = self.dropout(last_output)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        
        return x

# ==================== ÉTAPE 6: FONCTIONS D'ENTRAÎNEMENT ====================

def train_model(model, train_loader, val_loader, criterion, optimizer, epochs=100):
    """
    Entraîne le modèle avec validation
    """
    train_losses = []
    val_losses = []
    
    for epoch in range(epochs):
        # Phase d'entraînement
        model.train()
        train_loss = 0.0
        
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
        
        # Phase de validation
        model.eval()
        val_loss = 0.0
        
        with torch.no_grad():
            for batch_X, batch_y in val_loader:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                outputs = model(batch_X)
                loss = criterion(outputs, batch_y)
                val_loss += loss.item()
        
        # Calculer les pertes moyennes
        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)
        
        train_losses.append(avg_train_loss)
        val_losses.append(avg_val_loss)
        
        if (epoch + 1) % 10 == 0:
            print(f'Epoch [{epoch+1}/{epochs}] - Train Loss: {avg_train_loss:.6f}, Val Loss: {avg_val_loss:.6f}')
    
    return train_losses, val_losses

# ==================== ÉTAPE 7: ÉVALUATION ====================

def evaluate_model(model, test_loader, scaler_y):
    """
    Évalue le modèle et calcule le score R²
    """
    model.eval()
    predictions = []
    actuals = []
    
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            batch_X = batch_X.to(device)
            outputs = model(batch_X)
            predictions.extend(outputs.cpu().numpy())
            actuals.extend(batch_y.numpy())
    
    # Inverse scaling des prédictions
    predictions = np.array(predictions).reshape(-1, 1)
    actuals = np.array(actuals).reshape(-1, 1)
    
    predictions_original = scaler_y.inverse_transform(predictions)
    actuals_original = scaler_y.inverse_transform(actuals)
    
    # Calcul du R²
    r2 = r2_score(actuals_original, predictions_original)
    print(f"\nScore R² sur l'ensemble de test: {r2:.4f}")
    
    return predictions_original, actuals_original, r2

# ==================== ÉTAPE 8: VISUALISATION ====================

def plot_results(actual, predicted, train_losses, val_losses, dates=None):
    """
    Visualise les résultats du modèle
    """
    fig, axes = plt.subplots(2, 1, figsize=(15, 10))
    
    # Graphique des prédictions
    if dates is not None:
        axes[0].plot(dates, actual, label='Valeurs réelles', color='blue', alpha=0.7)
        axes[0].plot(dates, predicted, label='Prédictions', color='red', alpha=0.7)
        axes[0].set_xlabel('Date')
    else:
        axes[0].plot(actual, label='Valeurs réelles', color='blue', alpha=0.7)
        axes[0].plot(predicted, label='Prédictions', color='red', alpha=0.7)
        axes[0].set_xlabel('Échantillons')
    
    axes[0].set_ylabel('Prix de clôture')
    axes[0].set_title('Prédiction des cours boursiers - LSTM')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Graphique des pertes
    axes[1].plot(train_losses, label='Perte entraînement', color='blue')
    axes[1].plot(val_losses, label='Perte validation', color='orange')
    axes[1].set_xlabel('Époques')
    axes[1].set_ylabel('Perte (MSE)')
    axes[1].set_title('Courbes d\'apprentissage')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('stock_prediction_results.png', dpi=300, bbox_inches='tight')
    plt.show()

# ==================== ÉTAPE 9: SAUVEGARDE DU MODÈLE ====================

def save_model(model, scaler_X, scaler_y, model_path='lstm_model.pth'):
    """
    Sauvegarde le modèle et les scalers
    """
    torch.save({
        'model_state_dict': model.state_dict(),
        'model_architecture': {
            'input_size': 5,
            'hidden_size': 128,
            'num_layers': 2,
            'dropout': 0.2
        },
        'scaler_X': scaler_X,
        'scaler_y': scaler_y
    }, model_path)
    print(f"Modèle sauvegardé dans {model_path}")

# ==================== FONCTION PRINCIPALE ====================

def main():
    # 1. Charger et prétraiter les données
    print("="*50)
    print("CHARGEMENT ET PRÉTRAITEMENT DES DONNÉES")
    print("="*50)
    X, y, dates = load_and_preprocess_data(ticker='AAPL', start_date='2020-01-01', end_date='2023-12-31')
    
    # 2. Normaliser les données
    print("\n" + "="*50)
    print("NORMALISATION DES DONNÉES")
    print("="*50)
    X_scaled, y_scaled, scaler_X, scaler_y = normalize_data(X, y)
    
    # 3. Créer les séquences temporelles
    print("\n" + "="*50)
    print("CRÉATION DES SÉQUENCES TEMPORELLES")
    print("="*50)
    SEQ_LENGTH = 60
    X_seq, y_seq = create_sequences(X_scaled, y_scaled, seq_length=SEQ_LENGTH)
    
    # 4. Diviser les données
    print("\n" + "="*50)
    print("DIVISION DES DONNÉES")
    print("="*50)
    train_size = int(0.7 * len(X_seq))
    val_size = int(0.15 * len(X_seq))
    test_size = len(X_seq) - train_size - val_size
    
    X_train, y_train = X_seq[:train_size], y_seq[:train_size]
    X_val, y_val = X_seq[train_size:train_size+val_size], y_seq[train_size:train_size+val_size]
    X_test, y_test = X_seq[train_size+val_size:], y_seq[train_size+val_size:]
    
    print(f"Taille entraînement: {len(X_train)}")
    print(f"Taille validation: {len(X_val)}")
    print(f"Taille test: {len(X_test)}")
    
    # 5. Créer les DataLoaders
    batch_size = 64
    train_dataset = StockDataset(X_train, y_train)
    val_dataset = StockDataset(X_val, y_val)
    test_dataset = StockDataset(X_test, y_test)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    # 6. Définir le modèle
    print("\n" + "="*50)
    print("DÉFINITION DU MODÈLE LSTM")
    print("="*50)
    model = LSTMPredictor(
        input_size=5,  # Nombre de features
        hidden_size=128,
        num_layers=2,
        dropout=0.2
    ).to(device)
    
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    print(f"Architecture du modèle:\n{model}")
    print(f"Nombre de paramètres: {sum(p.numel() for p in model.parameters()):,}")
    
    # 7. Entraîner le modèle
    print("\n" + "="*50)
    print("ENTRAÎNEMENT DU MODÈLE")
    print("="*50)
    train_losses, val_losses = train_model(
        model, train_loader, val_loader, 
        criterion, optimizer, 
        epochs=100
    )
    
    # 8. Évaluer le modèle
    print("\n" + "="*50)
    print("ÉVALUATION DU MODÈLE")
    print("="*50)
    predictions, actuals, r2 = evaluate_model(model, test_loader, scaler_y)
    
    # 9. Visualiser les résultats
    print("\n" + "="*50)
    print("VISUALISATION DES RÉSULTATS")
    print("="*50)
    test_dates = dates[SEQ_LENGTH + train_size + val_size:]
    plot_results(actuals, predictions, train_losses, val_losses, test_dates)
    
    # 10. Sauvegarder le modèle
    print("\n" + "="*50)
    print("SAUVEGARDE DU MODÈLE")
    print("="*50)
    save_model(model, scaler_X, scaler_y)
    
    print("\n" + "="*50)
    print("EXÉCUTION TERMINÉE AVEC SUCCÈS!")
    print("="*50)

if __name__ == "__main__":
    main()

## Fonction pour charger un modèle sauvegardé et faire des prédictions

In [ ]:
def load_model_and_predict(model_path='lstm_model.pth', new_data=None):
    """
    Charge un modèle sauvegardé et fait des prédictions
    """
    # Chargement du checkpoint
    checkpoint = torch.load(model_path)
    
    # Reconstruire le modèle
    model = LSTMPredictor(**checkpoint['model_architecture'])
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    
    scaler_X = checkpoint['scaler_X']
    scaler_y = checkpoint['scaler_y']
    
    if new_data is not None:
        # Normaliser les nouvelles données
        new_data_scaled = scaler_X.transform(new_data)
        
        # Créer les séquences
        seq_length = 60
        if len(new_data_scaled) < seq_length:
            raise ValueError(f"Besoin d'au moins {seq_length} points de données")
        
        # Prendre les derniers seq_length points
        last_sequence = new_data_scaled[-seq_length:].reshape(1, seq_length, -1)
        
        # Faire la prédiction
        with torch.no_grad():
            prediction_scaled = model(torch.FloatTensor(last_sequence))
            prediction = scaler_y.inverse_transform(prediction_scaled.numpy())
        
        return prediction[0][0]
    
    return model

# Exemple d'utilisation
print("\n" + "="*50)
print("PRÉDICTION AVEC LE MODÈLE SAUVEGARDÉ")
print("="*50)

# Charger les nouvelles données
new_X, _, _ = load_and_preprocess_data(ticker='AAPL', start_date='2024-01-01', end_date='2024-02-01')
new_X_scaled, _, _, _ = normalize_data(new_X, np.zeros((len(new_X), 1)))

# Faire une prédiction
try:
    model = load_model_and_predict('lstm_model.pth')
    print("Modèle chargé avec succès!")
except:
    print("Aucun modèle sauvegardé trouvé. Exécutez d'abord la fonction principale.")